In [ ]:
%%sql

DROP TABLE IF EXISTS silver_rdm_session_status_add;

CREATE TABLE silver_rdm_session_status_add AS

WITH mpb_source AS (
    -- MPB source values derived from DRJ appointments and appointment attendances
    -- session_status_src_id uses same logic as session_status_src_name for MPB001

    SELECT DISTINCT
        CASE
            WHEN TRIM(a.cancelledBy) IS NOT NULL AND TRIM(a.cancelledBy) <> ''
                THEN CONCAT(TRIM(att.name), '_', TRIM(a.cancelledBy))
            ELSE TRIM(att.name)
        END AS session_status_src_name,

        LOWER(TRIM(
            CASE
                WHEN TRIM(a.cancelledBy) IS NOT NULL AND TRIM(a.cancelledBy) <> ''
                    THEN CONCAT(TRIM(att.name), '_', TRIM(a.cancelledBy))
                ELSE TRIM(att.name)
            END
        )) AS session_status_src_id,

        'MPB001' AS session_status_src_sys_inst_id

    FROM silver_drj_appointments a
    LEFT JOIN silver_drj_appointment_attendances att
        ON a.attendance_id = att.id
    WHERE att.name IS NOT NULL
      AND TRIM(att.name) <> ''
)

SELECT
    s.session_status_src_id,
    s.session_status_src_name,
    s.session_status_src_sys_inst_id
FROM mpb_source s
LEFT JOIN silver_rdm_session_status r
    ON LOWER(TRIM(s.session_status_src_id)) = LOWER(TRIM(r.session_status_src_id))
WHERE r.session_status_src_id IS NULL;

In [ ]:
%%sql
SELECT *
FROM silver_rdm_session_status_add
ORDER BY session_status_src_name;

In [ ]:
%%sql
SELECT COUNT(*) AS total_rows
FROM silver_rdm_session_status_add;

In [ ]:
%%sql
SELECT
    session_status_src_id,
    session_status_src_sys_inst_id,
    COUNT(*) AS cnt
FROM silver_rdm_session_status_add
GROUP BY session_status_src_id, session_status_src_sys_inst_id
HAVING COUNT(*) > 1;

old

In [ ]:
%%sql

DROP TABLE IF EXISTS silver_rdm_session_status_add;

CREATE TABLE silver_rdm_session_status_add AS

WITH mpb_source AS (
    SELECT DISTINCT
        CONCAT(COALESCE(TRIM(att.name), 'Unknown'), '_', COALESCE(TRIM(a.cancelledBy), 'Unknown')) AS session_status_src_name,
        LOWER(TRIM(CONCAT(COALESCE(TRIM(att.name), 'Unknown'), '_', COALESCE(TRIM(a.cancelledBy), 'Unknown')))) AS session_status_src_id,
        'MPB001' AS session_status_src_sys_inst_id
    FROM silver_drj_appointments a
    LEFT JOIN silver_drj_appointment_attendances att
        ON a.attendance_id = att.id
    WHERE (a.attendance_id IS NOT NULL OR a.cancelledBy IS NOT NULL)
      AND TRIM(CONCAT(COALESCE(TRIM(att.name), 'Unknown'), '_', COALESCE(TRIM(a.cancelledBy), 'Unknown'))) <> ''
)

SELECT
    s.session_status_src_id,
    s.session_status_src_name,
    s.session_status_src_sys_inst_id
FROM mpb_source s
LEFT JOIN silver_rdm_session_status r
    ON LOWER(TRIM(s.session_status_src_id)) = LOWER(TRIM(r.session_status_src_id))
WHERE r.session_status_src_id IS NULL;

In [ ]:
Old MPB logic used COALESCE(..., 'Unknown') for both attendance name and cancelledBy.

Reason:
- to avoid nulls during concatenation
- to always generate a value for session_status_src_name and session_status_src_id
- to keep rows where one side of the source combination was missing

Effect:
- rows such as Unknown_<guid> and Attended_Unknown were generated
- row count was higher because fallback values were retained

In [ ]:
Old MPB logic:
Used COALESCE(attendance_name,'Unknown') and COALESCE(cancelledBy,'Unknown') when deriving session_status_src_name and session_status_src_id.

Why:
To avoid null concatenation and retain rows even when one side of the source combination was missing.

Impact:
This produced fallback-derived values such as Unknown_<guid> and Attended_Unknown, which increased total row count.

New logic:
Removed the Unknown fallback, requires attendance name to exist, and appends cancelledBy only when present.
This reduced row count and produced cleaner values more aligned to Monday definition.